In [1]:
import requests

### SGLang API

##### Server Launch

In [2]:
# %%bash
# python -m sglang.launch_server \
#     --host 127.0.0.1 \
#     --port 30000 \
#     --model-path Qwen/Qwen2.5-7B-Instruct

##### Sending Requests

In [3]:
%%bash
curl http://127.0.0.1:30000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [
      {"role": "user", "content": "Efficient Deep Learning Systems course made me suffer."}
    ],
    "max_tokens": 128
  }'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

100  1312  100  1127  100   185    439     72  0:00:02  0:00:02 --:--:--   511


{"id":"94ac02ac12424ac0ae6bb8a96de94106","object":"chat.completion","created":1773717499,"model":"Qwen/Qwen2.5-7B-Instruct","choices":[{"index":0,"message":{"role":"assistant","content":"I'm sorry to hear that you're feeling this way about the Efficient Deep Learning Systems course! Learning new skills, especially in a complex field like deep learning, can be challenging and sometimes frustrating. Here are a few suggestions that might help you cope and even turn this experience into a more positive one:\n\n1. **Break It Down**: Sometimes, courses can feel overwhelming because they cover a lot of material. Try breaking down the course into smaller, manageable parts. Focus on understanding one concept at a time.\n\n2. **Practice Regularly**: Deep learning is a hands-on field, so try to practice coding and experimenting with the concepts you learn","reasoning_content":null,"tool_calls":null},"logprobs":null,"finish_reason":"length","matched_stop":null}],"usage":{"prompt_tokens":39,"total_

In [4]:
url = "http://127.0.0.1:30000/v1/chat/completions"

payload = {
    "model": "Qwen/Qwen2.5-7B-Instruct",
    "messages": [
        {"role": "user", "content": "Efficient Deep Learning Systems course made me suffer."}
    ],
    "max_tokens": 128
}

response = requests.post(url, json=payload)
print(response.json()["choices"][0]["message"]["content"])

I'm sorry to hear that you're feeling that way about the Efficient Deep Learning Systems course! Learning new and complex topics can be challenging, but it's also rewarding. Here are a few suggestions that might help you cope or even turn things around:

1. **Break It Down**: Try to break the course material into smaller, more manageable parts. Focus on understanding one concept at a time before moving on to the next.

2. **Practice**: Deep learning is as much about practice as it is about theory. Try implementing what you learn in small projects or exercises. This can help solidify your understanding and make the concepts more concrete.




### Inference Optimizations

##### Minimal Baseline

In [ ]:
# %%bash
# python -m sglang.launch_server \
#   --model-path Qwen/Qwen2.5-7B-Instruct \
#   --max-running-requests 1 \
#   --context-length 1024 \
#   --mem-fraction-static 0.5 \
#   --disable-radix-cache \
#   --disable-cuda-graph \
#   --sampling-backend pytorch \
#   --attention-backend triton

In [6]:
%%bash
python -m sglang.bench_serving \
  --backend sglang \
  --model Qwen/Qwen2.5-7B-Instruct \
  --num-prompts 100 \
  --sharegpt-output-len 100

benchmark_args=Namespace(backend='sglang', base_url=None, host='0.0.0.0', port=None, dataset_name='sharegpt', dataset_path='', model='Qwen/Qwen2.5-7B-Instruct', served_model_name=None, tokenizer=None, num_prompts=100, sharegpt_output_len=100, sharegpt_context_len=None, random_input_len=1024, random_output_len=1024, random_range_ratio=0.0, image_count=1, image_resolution='1080p', random_image_count=False, image_format='jpeg', image_content='random', request_rate=inf, use_trace_timestamps=False, max_concurrency=None, output_file=None, output_details=False, print_requests=False, disable_tqdm=False, disable_stream=False, return_logprob=False, return_routed_experts=False, seed=1, disable_ignore_eos=False, extra_request_body=None, apply_chat_template=False, profile=False, plot_throughput=False, profile_activities=['CPU', 'GPU'], profile_num_steps=None, profile_by_stage=False, profile_stages=None, profile_output_dir=None, profile_prefix=None, lora_name=None, lora_request_distribution='uniform

100%|██████████| 100/100 [03:07<00:00,  1.87s/it]



============ Serving Benchmark Result ============
Backend:                                 sglang    
Traffic request rate:                    inf       
Max request concurrency:                 not set   
Successful requests:                     100       
Benchmark duration (s):                  187.34    
Total input tokens:                      34100     
Total input text tokens:                 34100     
Total generated tokens:                  10000     
Total generated tokens (retokenized):    9999      
Request throughput (req/s):              0.53      
Input token throughput (tok/s):          182.02    
Output token throughput (tok/s):         53.38     
Peak output token throughput (tok/s):    57.00     
Peak concurrent requests:                100       
Total token throughput (tok/s):          235.40    
Concurrency:                             50.47     
----------------End-to-End Latency----------------
Mean E2E Latency (ms):                   94544.63  
Median E2E La

##### $+$ Continuous Batching

In [7]:
# %%bash
# python -m sglang.launch_server \
#   --model-path Qwen/Qwen2.5-7B-Instruct \
#   --max-running-requests 64 \
#   --context-length 1024 \
#   --mem-fraction-static 0.6 \
#   --disable-radix-cache \
#   --disable-cuda-graph \
#   --sampling-backend pytorch \
#   --attention-backend triton

In [8]:
%%bash
python -m sglang.bench_serving \
  --backend sglang \
  --model Qwen/Qwen2.5-7B-Instruct \
  --num-prompts 100 \
  --sharegpt-output-len 100

benchmark_args=Namespace(backend='sglang', base_url=None, host='0.0.0.0', port=None, dataset_name='sharegpt', dataset_path='', model='Qwen/Qwen2.5-7B-Instruct', served_model_name=None, tokenizer=None, num_prompts=100, sharegpt_output_len=100, sharegpt_context_len=None, random_input_len=1024, random_output_len=1024, random_range_ratio=0.0, image_count=1, image_resolution='1080p', random_image_count=False, image_format='jpeg', image_content='random', request_rate=inf, use_trace_timestamps=False, max_concurrency=None, output_file=None, output_details=False, print_requests=False, disable_tqdm=False, disable_stream=False, return_logprob=False, return_routed_experts=False, seed=1, disable_ignore_eos=False, extra_request_body=None, apply_chat_template=False, profile=False, plot_throughput=False, profile_activities=['CPU', 'GPU'], profile_num_steps=None, profile_by_stage=False, profile_stages=None, profile_output_dir=None, profile_prefix=None, lora_name=None, lora_request_distribution='uniform

100%|██████████| 100/100 [00:06<00:00, 14.95it/s]



============ Serving Benchmark Result ============
Backend:                                 sglang    
Traffic request rate:                    inf       
Max request concurrency:                 not set   
Successful requests:                     100       
Benchmark duration (s):                  6.70      
Total input tokens:                      34100     
Total input text tokens:                 34100     
Total generated tokens:                  10000     
Total generated tokens (retokenized):    9398      
Request throughput (req/s):              14.93     
Input token throughput (tok/s):          5091.44   
Output token throughput (tok/s):         1493.09   
Peak output token throughput (tok/s):    3008.00   
Peak concurrent requests:                100       
Total token throughput (tok/s):          6584.53   
Concurrency:                             67.75     
----------------End-to-End Latency----------------
Mean E2E Latency (ms):                   4537.88   
Median E2E La

##### $+$ Radix Cache & LPM Scheduling

In [ ]:
# %%bash
# python -m sglang.launch_server \
#   --model-path Qwen/Qwen2.5-7B-Instruct \
#   --max-running-requests 64 \
#   --schedule-policy lpm \
#   --context-length 1024 \
#   --mem-fraction-static 0.75 \
#   --disable-cuda-graph \
#   --sampling-backend pytorch \
#   --attention-backend triton

In [10]:
%%bash
python -m sglang.bench_serving \
  --backend sglang \
  --model Qwen/Qwen2.5-7B-Instruct \
  --num-prompts 100 \
  --sharegpt-output-len 100

benchmark_args=Namespace(backend='sglang', base_url=None, host='0.0.0.0', port=None, dataset_name='sharegpt', dataset_path='', model='Qwen/Qwen2.5-7B-Instruct', served_model_name=None, tokenizer=None, num_prompts=100, sharegpt_output_len=100, sharegpt_context_len=None, random_input_len=1024, random_output_len=1024, random_range_ratio=0.0, image_count=1, image_resolution='1080p', random_image_count=False, image_format='jpeg', image_content='random', request_rate=inf, use_trace_timestamps=False, max_concurrency=None, output_file=None, output_details=False, print_requests=False, disable_tqdm=False, disable_stream=False, return_logprob=False, return_routed_experts=False, seed=1, disable_ignore_eos=False, extra_request_body=None, apply_chat_template=False, profile=False, plot_throughput=False, profile_activities=['CPU', 'GPU'], profile_num_steps=None, profile_by_stage=False, profile_stages=None, profile_output_dir=None, profile_prefix=None, lora_name=None, lora_request_distribution='uniform

100%|██████████| 100/100 [00:04<00:00, 24.82it/s]



============ Serving Benchmark Result ============
Backend:                                 sglang    
Traffic request rate:                    inf       
Max request concurrency:                 not set   
Successful requests:                     100       
Benchmark duration (s):                  4.04      
Total input tokens:                      34100     
Total input text tokens:                 34100     
Total generated tokens:                  10000     
Total generated tokens (retokenized):    9398      
Request throughput (req/s):              24.76     
Input token throughput (tok/s):          8443.86   
Output token throughput (tok/s):         2476.21   
Peak output token throughput (tok/s):    4700.00   
Peak concurrent requests:                100       
Total token throughput (tok/s):          10920.07  
Concurrency:                             88.53     
----------------End-to-End Latency----------------
Mean E2E Latency (ms):                   3575.41   
Median E2E La

##### $+$ CUDA Graph Capture

In [ ]:
# %%bash
# python -m sglang.launch_server \
#   --model-path Qwen/Qwen2.5-7B-Instruct \
#   --max-running-requests 64 \
#   --schedule-policy lpm \
#   --context-length 1024 \
#   --mem-fraction-static 0.8 \
#   --sampling-backend pytorch \
#   --attention-backend triton

In [12]:
%%bash
python -m sglang.bench_serving \
  --backend sglang \
  --model Qwen/Qwen2.5-7B-Instruct \
  --num-prompts 100 \
  --sharegpt-output-len 100

benchmark_args=Namespace(backend='sglang', base_url=None, host='0.0.0.0', port=None, dataset_name='sharegpt', dataset_path='', model='Qwen/Qwen2.5-7B-Instruct', served_model_name=None, tokenizer=None, num_prompts=100, sharegpt_output_len=100, sharegpt_context_len=None, random_input_len=1024, random_output_len=1024, random_range_ratio=0.0, image_count=1, image_resolution='1080p', random_image_count=False, image_format='jpeg', image_content='random', request_rate=inf, use_trace_timestamps=False, max_concurrency=None, output_file=None, output_details=False, print_requests=False, disable_tqdm=False, disable_stream=False, return_logprob=False, return_routed_experts=False, seed=1, disable_ignore_eos=False, extra_request_body=None, apply_chat_template=False, profile=False, plot_throughput=False, profile_activities=['CPU', 'GPU'], profile_num_steps=None, profile_by_stage=False, profile_stages=None, profile_output_dir=None, profile_prefix=None, lora_name=None, lora_request_distribution='uniform

100%|██████████| 100/100 [00:03<00:00, 30.31it/s]



============ Serving Benchmark Result ============
Backend:                                 sglang    
Traffic request rate:                    inf       
Max request concurrency:                 not set   
Successful requests:                     100       
Benchmark duration (s):                  3.31      
Total input tokens:                      34100     
Total input text tokens:                 34100     
Total generated tokens:                  10000     
Total generated tokens (retokenized):    9398      
Request throughput (req/s):              30.22     
Input token throughput (tok/s):          10305.58  
Output token throughput (tok/s):         3022.16   
Peak output token throughput (tok/s):    6501.00   
Peak concurrent requests:                100       
Total token throughput (tok/s):          13327.74  
Concurrency:                             92.60     
----------------End-to-End Latency----------------
Mean E2E Latency (ms):                   3063.88   
Median E2E La

##### $+$ Speculative Decoding

In [ ]:
# %%bash
# python -m sglang.launch_server \
#   --model-path Qwen/Qwen2.5-7B-Instruct \
#   --context-length 1024 \
#   --max-running-requests 8 \
#   --mem-fraction-static 0.45 \
#   --schedule-policy lpm \
#   --sampling-backend pytorch \
#   --attention-backend triton \
#   --speculative-algorithm EAGLE \
#   --speculative-num-steps 2 \
#   --speculative-eagle-topk 2 \
#   --speculative-num-draft-tokens 4

In [14]:
%%bash
python -m sglang.bench_serving \
  --backend sglang \
  --model Qwen/Qwen2.5-7B-Instruct \
  --num-prompts 100 \
  --sharegpt-output-len 100

benchmark_args=Namespace(backend='sglang', base_url=None, host='0.0.0.0', port=None, dataset_name='sharegpt', dataset_path='', model='Qwen/Qwen2.5-7B-Instruct', served_model_name=None, tokenizer=None, num_prompts=100, sharegpt_output_len=100, sharegpt_context_len=None, random_input_len=1024, random_output_len=1024, random_range_ratio=0.0, image_count=1, image_resolution='1080p', random_image_count=False, image_format='jpeg', image_content='random', request_rate=inf, use_trace_timestamps=False, max_concurrency=None, output_file=None, output_details=False, print_requests=False, disable_tqdm=False, disable_stream=False, return_logprob=False, return_routed_experts=False, seed=1, disable_ignore_eos=False, extra_request_body=None, apply_chat_template=False, profile=False, plot_throughput=False, profile_activities=['CPU', 'GPU'], profile_num_steps=None, profile_by_stage=False, profile_stages=None, profile_output_dir=None, profile_prefix=None, lora_name=None, lora_request_distribution='uniform

100%|██████████| 100/100 [00:25<00:00,  3.87it/s]



============ Serving Benchmark Result ============
Backend:                                 sglang    
Traffic request rate:                    inf       
Max request concurrency:                 not set   
Successful requests:                     100       
Benchmark duration (s):                  25.83     
Total input tokens:                      34100     
Total input text tokens:                 34100     
Total generated tokens:                  10000     
Total generated tokens (retokenized):    9598      
Request throughput (req/s):              3.87      
Input token throughput (tok/s):          1320.03   
Output token throughput (tok/s):         387.11    
Peak output token throughput (tok/s):    495.00    
Peak concurrent requests:                100       
Total token throughput (tok/s):          1707.14   
Concurrency:                             52.10     
Accept length:                           2.69      
----------------End-to-End Latency----------------
Mean E2E Late

### Structured Generation